# Predicting late shipments with a neural network

Same dataset and same split as `01_naive_bayes.ipynb`: E-Commerce Shipping Data, target
`Reached.on.Time_Y.N` (1 = did NOT arrive on time).

Naive Bayes assumes the features are independent given the class, which is almost never true. A
multilayer perceptron makes no such assumption, so it can pick up interactions between features.
This notebook builds one, tunes it, and reports train and test performance.

## Step 1: Load the data

Same file as the previous notebook.

In [1]:
import pandas as pd

df = pd.read_csv('Train.csv')

print(df.shape)
df.head()

(10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


## Step 2: Preprocessing and split

1. Split 80/20 first, stratified, `random_state=42` - **identical to the previous notebook**, so
   the two models are evaluated on exactly the same test rows.
2. One-hot encode the four categorical columns, fit on train only.
3. Scale the numeric columns with `StandardScaler`, fit on train only.

The scaling step is new. `Weight_in_gms` runs into the thousands while `Customer_rating` is 1-5.
Gradient descent handles features on a similar scale far better than features that differ by three
orders of magnitude, so each numeric column is centred at 0 with standard deviation 1. Naive Bayes
did not need this because it fits a separate distribution per feature.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['ID', 'Reached.on.Time_Y.N'])
y = df['Reached.on.Time_Y.N']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
num_cols = [c for c in X_train.columns if c not in cat_cols]

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[cat_cols])
train_cat = encoder.transform(X_train[cat_cols])
test_cat = encoder.transform(X_test[cat_cols])

scaler = StandardScaler()
scaler.fit(X_train[num_cols])
train_num = scaler.transform(X_train[num_cols])
test_num = scaler.transform(X_test[num_cols])

X_train_final = np.hstack([train_num, train_cat])
X_test_final = np.hstack([test_num, test_cat])
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print('X_train_final:', X_train_final.shape)
print('X_test_final :', X_test_final.shape)

X_train_final: (8799, 19)
X_test_final : (2200, 19)


## Step 3: Build the network

The data is tabular, not images or sequences, so a plain feedforward MLP is the right shape.

`build_model()` is a function rather than a fixed model so Step 4 can call it repeatedly with
different hyperparameters.

- Hidden layers use `relu`.
- The output is a single neuron with `sigmoid`, since the target is binary. The output reads as
  the probability that the order is late.
- Loss is `binary_crossentropy`, optimiser is `Adam`.
- `l1_l2` regularisation on the hidden weights, to keep the network from memorising the training
  set.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Fix random seeds to reduce run-to-run variation
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

def build_model(input_dim, n_hidden_layers, units, reg_strength, lr=1e-3):
    reg = regularizers.l1_l2(l1=reg_strength, l2=reg_strength)

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(units, activation='relu', kernel_regularizer=reg))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

2026-08-06 23:01:48.336836: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Step 4: Hyperparameter search

Three hyperparameters, two values each:

- `hidden_layers`: 1 or 2
- `units`: 8 or 16
- `reg_strength`: 1e-4 or 1e-3 (weaker and stronger regularisation)

That is 8 combinations. The test set stays untouched - a validation set is carved out of the
training set instead, and each combination is scored on that. Tuning against the test set would
make the final test accuracy meaningless.

In [4]:
from sklearn.model_selection import train_test_split as tts2

grid_hidden_layers = [1, 2]
grid_units = [8, 16]
grid_reg_strength = [1e-4, 1e-3]

EPOCHS = 30
BATCH_SIZE = 32

X_tr, X_val, y_tr, y_val = tts2(
    X_train_final, y_train, test_size=0.2, stratify=y_train, random_state=42
)

input_dim = X_tr.shape[1]

best_val_acc = -1.0
best_params = None
results = []

for n_layers in grid_hidden_layers:
    for units in grid_units:
        for reg_strength in grid_reg_strength:
            tf.keras.backend.clear_session()

            model = build_model(input_dim, n_layers, units, reg_strength)
            history = model.fit(
                X_tr, y_tr,
                validation_data=(X_val, y_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0
            )

            val_acc = max(history.history['val_accuracy'])
            results.append((n_layers, units, reg_strength, val_acc))

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_params = (n_layers, units, reg_strength)

print('All results (hidden_layers, units, reg_strength, val_accuracy):')
for r in results:
    print(r)

print()
print('Best params (hidden_layers, units, reg_strength):', best_params)
print('Best validation accuracy:', best_val_acc)

All results (hidden_layers, units, reg_strength, val_accuracy):
(1, 8, 0.0001, 0.6607954502105713)
(1, 8, 0.001, 0.6625000238418579)
(1, 16, 0.0001, 0.668749988079071)
(1, 16, 0.001, 0.6727272868156433)
(2, 8, 0.0001, 0.6630681753158569)
(2, 8, 0.001, 0.6767045259475708)
(2, 16, 0.0001, 0.6670454740524292)
(2, 16, 0.001, 0.6778408885002136)

Best params (hidden_layers, units, reg_strength): (2, 16, 0.001)
Best validation accuracy: 0.6778408885002136


## Step 5: Train the final model

The winning combination is rebuilt and trained on the **full** training set, not just the reduced
portion used during the search, then evaluated on the training set and the held-out test set.

In [5]:
best_n_layers, best_units, best_reg_strength = best_params

tf.keras.backend.clear_session()
final_model = build_model(X_train_final.shape[1], best_n_layers, best_units, best_reg_strength)

final_model.fit(
    X_train_final, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0
)

train_loss, train_acc = final_model.evaluate(X_train_final, y_train, verbose=0)
test_loss, test_acc = final_model.evaluate(X_test_final, y_test, verbose=0)

print('Training Loss:', train_loss, ' Training Accuracy:', train_acc)
print('Test Loss:    ', test_loss, ' Test Accuracy:    ', test_acc)

Training Loss: 0.5215739607810974  Training Accuracy: 0.6865552663803101
Test Loss:     0.5265704393386841  Test Accuracy:     0.6709091067314148


## Step 6: Reading the result

The 8 combinations landed within a narrow band of each other, so the architecture choice mattered
less than expected here. The best was 2 hidden layers of 16 neurons with regularisation 0.001.

Test accuracy is 67.1%, against 64.4% for GaussianNB on the same split. The train/test accuracy
gap is 1.6 percentage points, so the network is not overfitting - it has simply reached what these
features support.

Two very different models landing within three points of each other is itself a result.
`03_comparison.ipynb` looks at what that means.

In [6]:
acc_gap_pp = (train_acc - test_acc) * 100
overfit_note = "very little" if abs(acc_gap_pp) < 3 else "some noticeable"

print(f"Architecture: input layer (19 features) -> {best_n_layers} hidden layer(s) of "
      f"{best_units} neurons each (relu, L1/L2 regularization = {best_reg_strength}) "
      f"-> 1 output neuron (sigmoid).")
print(f"Optimizer: Adam. Loss: binary_crossentropy. Trained for {EPOCHS} epochs.")
print()
print(f"{'':10}{'Loss':>10}{'Accuracy':>12}")
print(f"{'Train':10}{train_loss:>10.4f}{train_acc:>12.4f}")
print(f"{'Test':10}{test_loss:>10.4f}{test_acc:>12.4f}")
print()
print(f"Train vs test accuracy gap: {acc_gap_pp:.1f} percentage points -> "
      f"the model shows {overfit_note} overfitting.")

Architecture: input layer (19 features) -> 2 hidden layer(s) of 16 neurons each (relu, L1/L2 regularization = 0.001) -> 1 output neuron (sigmoid).
Optimizer: Adam. Loss: binary_crossentropy. Trained for 30 epochs.

                Loss    Accuracy
Train         0.5216      0.6866
Test          0.5266      0.6709

Train vs test accuracy gap: 1.6 percentage points -> the model shows very little overfitting.
